# 基础 DeepResearch 推理入口

本 notebook 是基础任务的便捷入口，对应代码位于 `core/agent/`。实际推理脚本是 `agent/run_deep_research.py`，评估脚本是 `agent/eval.py`。

使用前请先启动本地 vLLM 服务，并确保 BM25 index 已经存在。模型权重和数据集不包含在本提交包中。

In [ ]:
from pathlib import Path
import os

# 让 notebook 可从提交包根目录或 core/ 目录启动。
if Path("core/agent").exists():
    os.chdir("core")
elif not Path("agent").exists():
    raise RuntimeError("请从提交包根目录或 core/ 目录启动 notebook")

Path("../runs").mkdir(exist_ok=True)
Path("../indexes").mkdir(exist_ok=True)
print("Working directory:", Path.cwd())

## 路径配置

默认假设数据集和索引位于提交包根目录的同级位置：`../browsecomp_plus_hard50.jsonl` 与 `../indexes/browsecomp_plus_bm25.sqlite`。如果助教环境路径不同，只需要修改下面两个变量。

In [ ]:
DATASET = "../browsecomp_plus_hard50.jsonl"
INDEX_PATH = "../indexes/browsecomp_plus_bm25.sqlite"
MODEL = "qwen_auto"
BASE_URL = "http://127.0.0.1:8000/v1"
SUBMISSION = "../runs/deep_research_submission_v6_docref.jsonl"
EVAL_OUTPUT = "../runs/deep_research_eval_v6_docref.jsonl"

print("Dataset:", DATASET)
print("Index:", INDEX_PATH)
print("Submission:", SUBMISSION)
print("Eval output:", EVAL_OUTPUT)

## 可选：构建 BM25 索引

如果 `indexes/browsecomp_plus_bm25.sqlite` 已经存在，不需要运行本节。若索引缺失，请把 `CORPUS_PATH` 改成课程语料目录后取消最后一行注释运行。

In [ ]:
CORPUS_PATH = "../browsecomp-plus-corpus"
# !python -m agent.build_bm25_index --corpus-path {CORPUS_PATH} --index-path {INDEX_PATH} --overwrite

## 运行基础 DeepResearch agent

该配置对应提交包中外层 `eval/` 的 baseline 版本，评测准确率为 14%。

In [ ]:
!python -m agent.run_deep_research \
  --dataset {DATASET} \
  --index-path {INDEX_PATH} \
  --model {MODEL} \
  --base-url {BASE_URL} \
  --output {SUBMISSION}

## 评估基础任务结果

评估脚本保持课程版本，不在本提交中修改。

In [ ]:
!python -m agent.eval \
  --submission {SUBMISSION} \
  --dataset {DATASET} \
  --model {MODEL} \
  --base-url {BASE_URL} \
  --output {EVAL_OUTPUT}